# Raport 2 - model training

## Train test split - own implementation

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [ ]:
def stratified_train_val_test_split(X, y, test_size=0.2, val_size=0.1, random_state=42):
    # Accept pandas objects or numpy arrays
    if hasattr(X, 'values'):
        X = X.values
    if hasattr(y, 'values'):
        y = y.values

    classes = np.unique(y)
    train_indices = []
    val_indices = []
    test_indices = []

    rng = np.random.RandomState(random_state)

    for cls in classes:
        cls_idx = np.where(y == cls)[0].tolist()
        rng.shuffle(cls_idx)
        n = len(cls_idx)
        n_test = int(round(n * test_size))
        n_val = int(round(n * val_size))

        test_idx = cls_idx[:n_test]
        val_idx = cls_idx[n_test:n_test + n_val]
        train_idx = cls_idx[n_test + n_val:]

        train_indices.extend(train_idx)
        val_indices.extend(val_idx)
        test_indices.extend(test_idx)

    # Convert to numpy arrays for indexing
    train_indices = np.array(train_indices, dtype=int)
    val_indices = np.array(val_indices, dtype=int)
    test_indices = np.array(test_indices, dtype=int)

    X_train = X[train_indices]
    X_val = X[val_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_val = y[val_indices]
    y_test = y[test_indices]

    return X_train, X_val, X_test, y_train, y_val, y_test

df = pd.read_csv('alzheimers_disease_data.csv')
target = 'Diagnosis'
X = df.drop(columns=['PatientID', 'DoctorInCharge', target])
y = df[target].astype(int)

X_train, X_val, X_test, y_train, y_val, y_test = stratified_train_val_test_split(
    X, y, test_size=0.2, val_size=0.1, random_state=42
)


### Test the split balance 

In [4]:
print("\nDataset distribution:")
unique, counts = np.unique(y, return_counts=True)
total = len(y)
for cls, count in zip(unique, counts):
    percent = (count / total) * 100
    print(f"Class {cls}: {count} samples ({percent:.2f}%)")

unique_train, counts_train = np.unique(y_train, return_counts=True)
total_train = len(y_train)
print("\nTraining set distribution:")
for cls, count in zip(unique_train, counts_train):
    percent = (count / total_train) * 100
    print(f"Class {cls}: {count} samples ({percent:.2f}%)")

unique_val, counts_val = np.unique(y_val, return_counts=True)
total_val = len(y_val)
print("\nValidation set distribution:")
for cls, count in zip(unique_val, counts_val):
    percent = (count / total_val) * 100
    print(f"Class {cls}: {count} samples ({percent:.2f}%)")

unique_test, counts_test = np.unique(y_test, return_counts=True)
total_test = len(y_test)
print("\nTest set distribution:")
for cls, count in zip(unique_test, counts_test):
    percent = (count / total_test) * 100
    print(f"Class {cls}: {count} samples ({percent:.2f}%)")



Dataset distribution:
Class 0: 1389 samples (64.63%)
Class 1: 760 samples (35.37%)

Training set distribution:
Class 0: 972 samples (64.63%)
Class 1: 532 samples (35.37%)

Validation set distribution:
Class 0: 139 samples (64.65%)
Class 1: 76 samples (35.35%)

Test set distribution:
Class 0: 278 samples (64.65%)
Class 1: 152 samples (35.35%)


Restore the preprocessed datasets from files

In [36]:
import os

restore_folder = "scaled_data"
restored_datasets = {}

for filename in os.listdir(restore_folder):
    if filename.endswith(".csv"):
        key = filename.replace(".csv", "")
        df = pd.read_csv(f"{restore_folder}/{filename}")
        restored_datasets[key] = df

print("\nRestored datasets keys:\n")
for k, _ in restored_datasets.items():
    print( k)

scaled_datasets = restored_datasets | {'base': df}


Restored datasets keys:

constant_minmax
constant_zscore
interpolated_minmax
interpolated_zscore
knn_minmax
knn_zscore
pycaret_mean_minmax
pycaret_mean_zscore


## Train 3 scikit learn models and 2 external ones 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

sklearn_models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM': SVC(random_state=42)
}

external_models = {
    'XGBoost': XGBClassifier(random_state=42),
    # 'LightGBM': LGBMClassifier(random_state=42)  # Uncomment if LightGBM is installed
}

models = sklearn_models | external_models
models

{'Logistic Regression': LogisticRegression(random_state=42),
 'Random Forest': RandomForestClassifier(n_jobs=-1, random_state=42),
 'SVM': SVC(random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...)}

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Train models for each scaled dataset
all_results = {}

for dataset_name, df in scaled_datasets.items():
    print(f"\n{'='*70}")
    print(f"Processing dataset: {dataset_name}")
    print(f"{'='*70}")
    
    target = 'Diagnosis'
    X = df.drop(columns=[target])
    y = df[target].astype(int)
    
    X_train, X_val, X_test, y_train, y_val, y_test = stratified_train_val_test_split(
        X, y, test_size=0.2, val_size=0.1, random_state=42
    )
    
    dataset_results = {}
    
    # Train and evaluate each model
    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")
        
        model.fit(X_train, y_train)
        
        # Predictions on all sets
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)
        
        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
        test_recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
        
        dataset_results[model_name] = {
            'model': model,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'test_acc': test_acc,
            'precision': test_precision,
            'recall': test_recall,
            'f1': test_f1,
            'y_test_pred': y_test_pred
        }
        
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Validation Accuracy: {val_acc:.4f}")
    
    all_results[dataset_name] = dataset_results


Processing dataset: constant_minmax

Training Logistic Regression...
  Train Accuracy: 0.7753
  Validation Accuracy: 0.7767

Training Random Forest...
  Train Accuracy: 1.0000
  Validation Accuracy: 0.9023

Training SVM...
  Train Accuracy: 1.0000
  Validation Accuracy: 0.9023

Training SVM...
  Train Accuracy: 0.9102
  Validation Accuracy: 0.8000

Training XGBoost...
  Train Accuracy: 1.0000
  Validation Accuracy: 0.9116

Processing dataset: constant_zscore

Training Logistic Regression...
  Train Accuracy: 0.7753
  Validation Accuracy: 0.7721

Training Random Forest...
  Train Accuracy: 0.9102
  Validation Accuracy: 0.8000

Training XGBoost...
  Train Accuracy: 1.0000
  Validation Accuracy: 0.9116

Processing dataset: constant_zscore

Training Logistic Regression...
  Train Accuracy: 0.7753
  Validation Accuracy: 0.7721

Training Random Forest...
  Train Accuracy: 1.0000
  Validation Accuracy: 0.9023

Training SVM...
  Train Accuracy: 1.0000
  Validation Accuracy: 0.9023

Training S

In [38]:
print(f"\n{'='*70}")
print("FINAL SUMMARY - All Datasets and Models")
print(f"{'='*70}")
for dataset_name, dataset_results in all_results.items():
    print(f"\n{dataset_name}:")
    for model_name, metrics in dataset_results.items():
        print(f"  {model_name}: Accuracy={metrics['test_acc']:.4f}, F1={metrics['f1']:.4f}")


FINAL SUMMARY - All Datasets and Models

constant_minmax:
  Logistic Regression: Accuracy=0.7791, F1=0.7707
  Random Forest: Accuracy=0.8674, F1=0.8655
  SVM: Accuracy=0.7791, F1=0.7694
  XGBoost: Accuracy=0.8814, F1=0.8805

constant_zscore:
  Logistic Regression: Accuracy=0.7884, F1=0.7814
  Random Forest: Accuracy=0.8674, F1=0.8655
  SVM: Accuracy=0.7767, F1=0.7667
  XGBoost: Accuracy=0.8814, F1=0.8805

interpolated_minmax:
  Logistic Regression: Accuracy=0.7907, F1=0.7851
  Random Forest: Accuracy=0.8349, F1=0.8286
  SVM: Accuracy=0.7628, F1=0.7514
  XGBoost: Accuracy=0.8558, F1=0.8532

interpolated_zscore:
  Logistic Regression: Accuracy=0.7860, F1=0.7813
  Random Forest: Accuracy=0.8326, F1=0.8260
  SVM: Accuracy=0.7628, F1=0.7514
  XGBoost: Accuracy=0.8558, F1=0.8532

knn_minmax:
  Logistic Regression: Accuracy=0.8047, F1=0.7994
  Random Forest: Accuracy=0.8605, F1=0.8567
  SVM: Accuracy=0.8093, F1=0.8007
  XGBoost: Accuracy=0.8884, F1=0.8873

knn_zscore:
  Logistic Regression: 